In [2]:
import sys
from pathlib import Path

import pandas as pd
from pypdf import PdfReader


# Project root
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Testing PDF
PDF_PATH = (
    PROJECT_ROOT
    / "data"
    / "CL Adyen NV_Jurgen Geerts.pdf"
)

print("Project:", PROJECT_ROOT)
print("PDF:", PDF_PATH)
print("PDF exists:", PDF_PATH.exists())

Project: c:\Users\win10\Desktop\Agentic AI Course\GenAI_support\financial_spreading
PDF: c:\Users\win10\Desktop\Agentic AI Course\GenAI_support\financial_spreading\data\CL Adyen NV_Jurgen Geerts.pdf
PDF exists: True


In [3]:
from pypdf import PdfReader

reader = PdfReader(str(PDF_PATH))

selected_pages = []

for page_number, page in enumerate(reader.pages, start=1):

    text = page.extract_text() or ""

    selected_pages.append({
        "page_number": page_number,
        "page": page_number,
        "text": text,
    })

print(f"Total pages: {len(selected_pages)}")

display(
    pd.DataFrame([
        {
            "Page": page["page_number"],
            "Characters": len(page["text"]),
            "Words": len(page["text"].split()),
        }
        for page in selected_pages
    ])
)

Total pages: 9


,Page,Characters,Words
0,1,1654,189
1,2,766,105
2,3,1605,185
3,4,537,63
4,5,1390,160
5,6,363,44
6,7,1446,197
7,8,1417,194
8,9,59,7


In [4]:
from nodes.keyword_search import (
    clean_keywords,
    build_keywords,
    extract_context,
    find_keyword_matches,
    keyword_search,
)

print("keyword_search.py imported successfully")

keyword_search.py imported successfully


In [5]:
test_metrics = [
    {
        "worksheet": "Balance Sheet",
        "variable": "Accounts Receivable Trade",
        "description": "Trade receivables from customers",
        "remarks": "",
    },
    {
        "worksheet": "Balance Sheet",
        "variable": "Cash and Bank",
        "description": "Cash and bank balances",
        "remarks": "",
    },
    {
        "worksheet": "Balance Sheet",
        "variable": "Total Assets",
        "description": "Total assets",
        "remarks": "",
    },
    {
        "worksheet": "Income Statement",
        "variable": "Net Sales",
        "description": "Total revenue or net sales",
        "remarks": "",
    },
    {
        "worksheet": "Income Statement",
        "variable": "EBITDA",
        "description": "Earnings before interest taxes depreciation and amortization",
        "remarks": "",
    },
    {
        "worksheet": "Income Statement",
        "variable": "Net Income",
        "description": "Profit after income taxes",
        "remarks": "",
    },
    {
        "worksheet": "Cash Flow",
        "variable": "Net Cash From Operations",
        "description": "Cash generated from operating activities",
        "remarks": "",
    },
    {
        "worksheet": "Financial Ratios",
        "variable": "Current Ratio",
        "description": "Current assets divided by current liabilities",
        "remarks": "",
    },
]


keyword_rows = []

for metric in test_metrics:

    keywords = build_keywords(metric)

    keyword_rows.append({
        "Variable": metric["variable"],
        "Keyword Count": len(keywords),
        "Keywords": ", ".join(keywords),
    })


display(pd.DataFrame(keyword_rows))

,Variable,Keyword Count,Keywords
0,Accounts Receivable Trade,6,"accounts receivable trade, accounts, receivabl..."
1,Cash and Bank,4,"cash and bank, cash, bank, balances"
2,Total Assets,2,"total assets, assets"
3,Net Sales,4,"net sales, net, sales, revenue"
4,EBITDA,7,"ebitda, earnings, before, interest, taxes, dep..."
5,Net Income,6,"net income, net, income, profit, after, taxes"
6,Net Cash From Operations,7,"net cash from operations, net, cash, operation..."
7,Current Ratio,6,"current ratio, current, ratio, assets, divided..."


In [6]:
test_state = {
    "excel_data": test_metrics,
    "selected_pages": selected_pages,
}

keyword_result = keyword_search(test_state)

keyword_contexts = keyword_result["keyword_contexts"]

print(f"Metrics processed: {len(keyword_contexts)}")


# Summary
results = []

for item in keyword_contexts:

    pages = sorted({
        match["page_number"]
        for match in item["matches"]
    })

    results.append({
        "Variable": item["variable"],
        "Keywords": len(item["keywords"]),
        "Matches": len(item["matches"]),
        "Pages Found": pages,
    })


display(pd.DataFrame(results))

Metrics processed: 8


,Variable,Keywords,Matches,Pages Found
0,Accounts Receivable Trade,6,9,"[1, 2, 7]"
1,Cash and Bank,4,14,"[1, 2, 5, 6, 7, 8]"
2,Total Assets,2,22,"[1, 4, 5, 7, 8]"
3,Net Sales,4,34,"[1, 3, 4, 5, 7, 8]"
4,EBITDA,7,25,"[1, 3, 4, 5, 6, 7, 8]"
5,Net Income,6,43,"[1, 3, 4, 5, 7, 8]"
6,Net Cash From Operations,7,41,"[1, 3, 4, 5, 6, 7, 8]"
7,Current Ratio,6,32,"[1, 2, 4, 5, 7, 8]"


In [7]:
expected_pages = {
    "Accounts Receivable Trade": [1],
    "Cash and Bank": [1],
    "Total Assets": [1],
    "Net Sales": [3],
    "EBITDA": [3],
    "Net Income": [4],
    "Net Cash From Operations": [5],
    "Current Ratio": [7],
}


validation = []

for item in keyword_contexts:

    variable = item["variable"]

    actual_pages = sorted({
        match["page_number"]
        for match in item["matches"]
    })

    expected = expected_pages.get(variable, [])

    passed = bool(
        set(expected).intersection(actual_pages)
    )

    validation.append({
        "Variable": variable,
        "Expected": expected,
        "Found": actual_pages,
        "Matches": len(item["matches"]),
        "Status": "PASS" if passed else "FAIL",
    })


validation_df = pd.DataFrame(validation)

display(validation_df)


print("\n========== TEST SUMMARY ==========")

passed = (validation_df["Status"] == "PASS").sum()
failed = (validation_df["Status"] == "FAIL").sum()

print(f"Passed : {passed}")
print(f"Failed : {failed}")
print(f"Total  : {len(validation_df)}")

,Variable,Expected,Found,Matches,Status
0,Accounts Receivable Trade,[1],"[1, 2, 7]",9,PASS
1,Cash and Bank,[1],"[1, 2, 5, 6, 7, 8]",14,PASS
2,Total Assets,[1],"[1, 4, 5, 7, 8]",22,PASS
3,Net Sales,[3],"[1, 3, 4, 5, 7, 8]",34,PASS
4,EBITDA,[3],"[1, 3, 4, 5, 6, 7, 8]",25,PASS
5,Net Income,[4],"[1, 3, 4, 5, 7, 8]",43,PASS
6,Net Cash From Operations,[5],"[1, 3, 4, 5, 6, 7, 8]",41,PASS
7,Current Ratio,[7],"[1, 2, 4, 5, 7, 8]",32,PASS



========== TEST SUMMARY ==========
Passed : 8
Failed : 0
Total  : 8
